# Understand Retrieval Methods

This notebook compares the three retrieval methods in the project: BM25, dense retrieval, and hybrid retrieval. It focuses only on retrieval. No answer-generation model is called.

## 1. Setup

The notebook imports the project package from `src/`, so it can run from either the project root or the `notebooks/` directory.

In [1]:
from collections import Counter
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT

PosixPath('/Users/KnowYourself/Desktop/Desktop/Obsidian_Vault/Personal Assistant/Learn_Internship/CS6493_Group_Project/rag_project')

In [5]:
from rag_experiment.data.hotpotqa import load_hotpot_jsonl
from rag_experiment.model_clients import get_embedding_model
from rag_experiment.model_clients.providers import EMBEDDING_PROVIDERS
from rag_experiment.retrieval import BM25Retriever, DenseRetriever, HybridRetriever

DATA_PATH = PROJECT_ROOT / "data" / "hotpotqa_mini" / "sample.jsonl"
examples = load_hotpot_jsonl(DATA_PATH)
example = examples[0]
passages = example.passages()
# print(json.dumps(example.__dict__, indent=2, ensure_ascii=False))
print("question:", example.question)
print("gold answer:", example.answer)
print("context:", example.context)
print("passages:", len(passages))

question: Which university is located in the city where the CN Tower stands, the University of Toronto or McGill University?
gold answer: University of Toronto
context: (('CN Tower', ('The CN Tower is a communications and observation tower in Toronto, Ontario, Canada.', 'It was completed in 1976 and became a major landmark of the city.')), ('University of Toronto', ('The University of Toronto is a public research university in Toronto, Ontario, Canada.', "It is located on the grounds that surround Queen's Park.")), ('McGill University', ('McGill University is a public research university in Montreal, Quebec, Canada.', 'It was established by royal charter in 1821.')))
passages: 6


## 2. The Search Space

All three retrievers search over the same passages. In this tiny fixture, each sentence is one passage.

In [3]:
for passage in passages:
    print(f"{passage.id} | {passage.text}")

mini_001::CN Tower::0 | The CN Tower is a communications and observation tower in Toronto, Ontario, Canada.
mini_001::CN Tower::1 | It was completed in 1976 and became a major landmark of the city.
mini_001::University of Toronto::0 | The University of Toronto is a public research university in Toronto, Ontario, Canada.
mini_001::University of Toronto::1 | It is located on the grounds that surround Queen's Park.
mini_001::McGill University::0 | McGill University is a public research university in Montreal, Quebec, Canada.
mini_001::McGill University::1 | It was established by royal charter in 1821.


## 3. Helper for Reading Results

`score` means different things for different retrievers. BM25 currently has no exposed numeric score from LangChain. Dense uses vector similarity, where higher is closer. Hybrid uses LangChain EnsembleRetriever; fused scores are not exposed in our artifact wrapper.

In [4]:
def show_results(name, results):
    print(f"\n{name}")
    print("=" * len(name))
    for result in results:
        passage = result.passage
        print(f"rank={result.rank} score={result.score}")
        print(f"passage_id={passage.id}")
        print(f"title={passage.title} sentence_index={passage.sentence_index}")
        print(f"text={passage.text}")
        print(f"metadata={result.metadata}")
        print()

## 4. BM25 Retrieval

BM25 is lexical. It likes passages that share exact or near-exact words with the question.

In [5]:
top_k = 3
bm25_results = BM25Retriever(passages, top_k=top_k).retrieve(example.question)
show_results("BM25", bm25_results)


BM25
====
rank=1 score=None
passage_id=mini_001::University of Toronto::1
title=University of Toronto sentence_index=1
text=It is located on the grounds that surround Queen's Park.
metadata={'retriever': 'bm25'}

rank=2 score=None
passage_id=mini_001::University of Toronto::0
title=University of Toronto sentence_index=0
text=The University of Toronto is a public research university in Toronto, Ontario, Canada.
metadata={'retriever': 'bm25'}

rank=3 score=None
passage_id=mini_001::McGill University::0
title=McGill University sentence_index=0
text=McGill University is a public research university in Montreal, Quebec, Canada.
metadata={'retriever': 'bm25'}



## 5. DashScope Embedding API Smoke Test

Dense and hybrid retrieval need embeddings. This cell makes one real DashScope OpenAI-compatible embedding call, then prints only safe diagnostics: provider, model, base URL, vector length, and a tiny rounded preview.

In [6]:
embedding_provider = "dashscope"
embedding_cfg = EMBEDDING_PROVIDERS[embedding_provider]
api_key_env = embedding_cfg["api_key_env"]

if not os.getenv(api_key_env):
    raise RuntimeError(f"Set {api_key_env} before running dense retrieval.")

embedding_model = get_embedding_model(provider=embedding_provider)
sample_vector = embedding_model.embed_query(example.question)

print("provider:", embedding_provider)
print("model:", embedding_cfg["default_model"])
print("base_url:", embedding_cfg["base_url"])
print("vector_length:", len(sample_vector))
print("vector_preview:", [round(value, 6) for value in sample_vector[:5]])

provider: dashscope
model: text-embedding-v2
base_url: https://dashscope.aliyuncs.com/compatible-mode/v1
vector_length: 1536
vector_preview: [-0.049231, 0.026927, 0.030952, 0.018731, -0.00586]


## 6. Dense Retrieval

Dense retrieval embeds the question and passages, then searches in vector space. It reuses the embedding client from the API smoke test.

In [7]:
dense_results = DenseRetriever(
    passages,
    embedding_model=embedding_model,
    top_k=top_k,
).retrieve(example.question)
show_results("Dense", dense_results)


Dense
=====
rank=1 score=0.7416993182323579
passage_id=mini_001::CN Tower::0
title=CN Tower sentence_index=0
text=The CN Tower is a communications and observation tower in Toronto, Ontario, Canada.
metadata={'retriever': 'dense', 'score_kind': 'cosine_similarity'}

rank=2 score=0.7016335727430533
passage_id=mini_001::University of Toronto::0
title=University of Toronto sentence_index=0
text=The University of Toronto is a public research university in Toronto, Ontario, Canada.
metadata={'retriever': 'dense', 'score_kind': 'cosine_similarity'}

rank=3 score=0.5926655997179732
passage_id=mini_001::McGill University::0
title=McGill University sentence_index=0
text=McGill University is a public research university in Montreal, Quebec, Canada.
metadata={'retriever': 'dense', 'score_kind': 'cosine_similarity'}



## 7. Hybrid Retrieval

Hybrid retrieval combines BM25 and dense retrieval with LangChain EnsembleRetriever. EnsembleRetriever uses weighted reciprocal rank fusion, so BM25 and vector scores do not need to be directly comparable.

In [8]:
hybrid_bm25_retriever = BM25Retriever(passages, top_k=top_k)
hybrid_dense_retriever = DenseRetriever(
    passages,
    embedding_model=embedding_model,
    top_k=top_k,
)
hybrid_results = HybridRetriever(
    bm25_retriever=hybrid_bm25_retriever,
    dense_retriever=hybrid_dense_retriever,
    top_k=top_k,
    rrf_k=60,
).retrieve(example.question)
show_results("Hybrid Ensemble RRF", hybrid_results)


Hybrid Ensemble RRF
rank=1 score=None
passage_id=mini_001::University of Toronto::0
title=University of Toronto sentence_index=0
text=The University of Toronto is a public research university in Toronto, Ontario, Canada.
metadata={'retriever': 'hybrid', 'fusion': 'langchain_ensemble_rrf', 'weights': [0.5, 0.5], 'rrf_k': 60}

rank=2 score=None
passage_id=mini_001::University of Toronto::1
title=University of Toronto sentence_index=1
text=It is located on the grounds that surround Queen's Park.
metadata={'retriever': 'hybrid', 'fusion': 'langchain_ensemble_rrf', 'weights': [0.5, 0.5], 'rrf_k': 60}

rank=3 score=None
passage_id=mini_001::CN Tower::0
title=CN Tower sentence_index=0
text=The CN Tower is a communications and observation tower in Toronto, Ontario, Canada.
metadata={'retriever': 'hybrid', 'fusion': 'langchain_ensemble_rrf', 'weights': [0.5, 0.5], 'rrf_k': 60}



## 8. Quick Comparison

This compact view helps you see whether each retriever is returning the same evidence or different evidence.

In [9]:
for name, results in [
    ("bm25", bm25_results),
    ("dense", dense_results),
    ("hybrid", hybrid_results),
]:
    print(name)
    for result in results:
        print(f"  {result.rank}. {result.passage.id}")

bm25
  1. mini_001::University of Toronto::1
  2. mini_001::University of Toronto::0
  3. mini_001::McGill University::0
dense
  1. mini_001::CN Tower::0
  2. mini_001::University of Toronto::0
  3. mini_001::McGill University::0
hybrid
  1. mini_001::University of Toronto::0
  2. mini_001::University of Toronto::1
  3. mini_001::CN Tower::0


## 9. Saved Artifact Check

The command-line dry runs write JSONL artifacts. This cell checks that the saved files contain records, retrieval results, and no recorded errors.

In [10]:
artifact_paths = [
    PROJECT_ROOT / "outputs" / "hotpotqa_mini_bm25_dry_run" / "results.jsonl",
    PROJECT_ROOT / "outputs" / "hotpotqa_mini_dense_dry_run" / "results.jsonl",
    PROJECT_ROOT / "outputs" / "hotpotqa_mini_hybrid_dry_run" / "results.jsonl",
]

for artifact_path in artifact_paths:
    rows = [json.loads(line) for line in artifact_path.read_text().splitlines() if line.strip()]
    errors = [row["error"] for row in rows if row.get("error")]
    retrieved_counts = Counter(len(row.get("retrieved_passages", [])) for row in rows)
    first_metadata = None
    if rows and rows[0].get("retrieved_passages"):
        first_metadata = rows[0]["retrieved_passages"][0]["metadata"]

    print(artifact_path.relative_to(PROJECT_ROOT))
    print("  rows:", len(rows))
    print("  errors:", len(errors))
    print("  retrieved_passage_counts:", dict(sorted(retrieved_counts.items())))
    print("  first_metadata:", first_metadata)

outputs/hotpotqa_mini_bm25_dry_run/results.jsonl
  rows: 5
  errors: 0
  retrieved_passage_counts: {3: 5}
  first_metadata: {'retriever': 'bm25'}
outputs/hotpotqa_mini_dense_dry_run/results.jsonl
  rows: 5
  errors: 0
  retrieved_passage_counts: {3: 5}
  first_metadata: {'retriever': 'dense', 'score_kind': 'cosine_similarity'}
outputs/hotpotqa_mini_hybrid_dry_run/results.jsonl
  rows: 5
  errors: 0
  retrieved_passage_counts: {3: 5}
  first_metadata: {'retriever': 'hybrid', 'fusion': 'langchain_ensemble_rrf', 'weights': [0.5, 0.5], 'rrf_k': 60}


## Takeaway

BM25 is the lexical baseline. Dense retrieval uses semantic similarity from embeddings. Hybrid retrieval is a practical combination: it rewards passages that rank well in either or both retrievers.